# LAB03 · Métricas de negocio, análisis de logs y primeras gráficas

**Curso Big Data e IA Aplicada · Formación San Miguel**

---

### Cómo funciona este cuaderno

Este cuaderno **no te da todas las respuestas**. Te da los comandos, los números que deben salir, y
**preguntas que tienes que resolver tú**.

Cada pregunta lleva una etiqueta que dice **a quién se le pregunta**. Esa etiqueta es la lección
más importante del día:

| Etiqueta | A quién preguntas | Para qué |
|---|---|---|
| 🗂️ **RAG** | Al cuaderno de NotebookLM, con **el material del curso** | Conceptos y porqués. **Exígele que cite el apartado** |
| 🤖 **ASISTENTE** | A Gemini o ChatGPT | Sintaxis y comandos. Con el protocolo de cuatro pasos |
| ⚙️ **MÁQUINA** | A nadie: **se ejecuta** | Los números. Ningún modelo los sabe |

> **La regla que gobierna el día:** si la respuesta es un **número de tus datos**, no se le pregunta
> a ninguna IA. Se ejecuta. El RAG recupera y cita, el asistente propone sintaxis, **la máquina
> calcula, y tú verificas**.

### El protocolo de cuatro pasos (para las 🤖)

```
1. CONTEXTO PRIMERO    pega la cabecera y dos filas de ejemplo
2. EXIGE EXPLICACIÓN   el comando y su desglose pieza a pieza
3. EJECUTA SOBRE POCO  head -1000 | …  si hay error, que explote en pequeño
4. CONTRAPRUEBA        valida por una segunda vía antes de darlo por bueno
```

### Y si te atascas más de 5 minutos

```
«Estoy atascado en el [LAB y paso]. He ejecutado: [comando].
 Ha salido: [salida o error completo].
 1) Explícame qué significa.
 2) Dame una sola pista para avanzar, no la solución.
 3) Dime cómo comprobaré que funciona.»
```

Y si tras la pista sigues atascado: **mano arriba**.

---

### Las columnas de `ventas.csv`

```
1 id_venta · 2 fecha · 3 id_cliente · 4 id_producto · 5 categoria
6 unidades · 7 precio_unitario · 8 ciudad · 9 canal
```

### Las de `access.log` (separado por espacios, no por comas)

```
$1 IP · $4 $5 fecha · $6 metodo · $7 ruta · $8 protocolo · $9 codigo · $10 bytes
```

Este cuaderno vive en `notebooks/`. Los datos están un piso arriba: `../datasets/`

## 0 · Comprobación del entorno

In [ ]:
import os

print("Estoy en:", os.getcwd())
print()
for f in ["ventas.csv", "clientes.csv", "access.log", "productos.json"]:
    ruta = f"../datasets/{f}"
    if os.path.exists(ruta):
        print(f"  OK    {f:16s} {os.path.getsize(ruta)/1024/1024:7.1f} MB")
    else:
        print(f"  FALTA {f}  <-- avisa al docente")

print()
try:
    import matplotlib.pyplot as plt
    print("  OK    matplotlib disponible")
except ImportError:
    print("  FALTA matplotlib  ->  mamba install -y -c conda-forge matplotlib")

---
---

# PASO 1 · El contador de frecuencias

Es **la pieza estrella** del bloque. Léela como una frase:

> corta la columna → quítame la cabecera → junta los iguales → cuenta cada grupo → ordena por tamaño

In [ ]:
%%bash
cd ../datasets
cut -d, -f5 ventas.csv | tail -n +2 | sort | uniq -c | sort -rn

# Esperado:
#   200487 jardin · 200210 deporte · 200208 hogar · 199786 informatica · 199309 papeleria

### El experimento: ahora **sin** el `tail -n +2`

In [ ]:
%%bash
cd ../datasets
cut -d, -f5 ventas.csv | sort | uniq -c | sort -rn

### La misma tubería, otras dos columnas

In [ ]:
%%bash
cd ../datasets

echo "== CIUDADES (top 5) =="
cut -d, -f8 ventas.csv | tail -n +2 | sort | uniq -c | sort -rn | head -5

echo ""
echo "== CANALES =="
cut -d, -f9 ventas.csv | tail -n +2 | sort | uniq -c | sort -rn

# Ciudades: 337910 Zaragoza · 138663 Madrid · 118418 Barcelona · 98230 Huesca · 66824 Valencia
# Canales:  499412 tienda · 333400 web · 167188 movil
#
# APUNTA LOS TRES CANALES: el miércoles Spark tendrá que clavarlos.

---
---

# PASO 2 · Facturación total y ticket medio

Aquí dejamos de contar y empezamos a **calcular**.

```
awk -F, 'NR>1 {t += $6*$7; n++} END {printf "%.2f\n", t}' ventas.csv
     │    │      │                  │
     │    │      │                  └── una sola vez, al agotarse el fichero
     │    │      └── por CADA línea: unidades × precio, acumulado. n++ cuenta
     │    └── la condición: salta la cabecera
     └── los campos se separan por comas
```

In [ ]:
%%bash
cd ../datasets
awk -F, 'NR>1 {t += $6*$7; n++} END {printf "Total: %.2f  Ticket medio: %.2f\n", t, t/n}' ventas.csv

# Esperado: Total: 429888864.70   Ticket medio: 429.89

### ⚠️ La verificación — este paso NO se salta

Un índice de columna equivocado produce un resultado **perfectamente verosímil y perfectamente
falso**. Si pones `$6` donde va `$7`, tu facturación es un número precioso que no significa nada.

**La contraprueba en pequeño** consiste en validar la *mecánica* con datos que puedas comprobar con
los dedos. Tres filas bastan.

In [ ]:
%%bash
cd ../datasets

echo "== 1. Las tres primeras VENTAS: unidades y precio =="
head -4 ventas.csv | cut -d, -f6,7

echo ""
echo "== 2. Multiplícalas TÚ, a mano o con la calculadora =="
echo "        2 x 219.90  = ?"
echo "        1 x  34.50  = ?"
echo "        1 x  35.18  = ?"

echo ""
echo "== 3. Y ahora que las multiplique awk. ¿COINCIDEN? =="
awk -F, 'NR>1 {print $6 * $7}' ventas.csv | head -3

# Esperado:  439.8  ·  34.5  ·  35.18
#
# Si tus tres cuentas coinciden con las tres de awk, la mecánica es la correcta
# y YA PUEDES FIARTE del total del millón de filas. Eso es la contraprueba en pequeño:
# validar el METODO con datos que puedes comprobar con los dedos.

### 📉 Y una segunda comprobación, de otro tipo

Ahora las mismas métricas, pero **solo sobre las 1.000 primeras filas**.

⚠️ **Ojo: el ticket medio NO va a coincidir con el del fichero entero, y está bien que no coincida.**
Mil filas son una **muestra**; un millón es la **población**. Que una muestra pequeña dé un promedio
distinto no es un error: es muestreo.

Lo que sí tiene que salir bien es **la mecánica** — y eso ya lo has validado arriba.

In [ ]:
%%bash
cd ../datasets

echo "== Sobre las 1.000 primeras filas =="
head -1001 ventas.csv | awk -F, 'NR>1 {t += $6*$7; n++} END {printf "  filas=%d   total=%.2f   ticket=%.2f\n", n, t, t/n}'

echo ""
echo "== Sobre el fichero ENTERO =="
awk -F, 'NR>1 {t += $6*$7; n++} END {printf "  filas=%d   total=%.2f   ticket=%.2f\n", n, t, t/n}' ventas.csv

echo ""
echo "Los tickets son distintos. ¿Te preocupa? No debería: 1.000 filas son una MUESTRA."

---
---

# 🔍 CONSULTA 1 · Bloque A

**15 minutos.** Resuelve estas siete. Las 🗂️ y 🤖 con tu IA; las ⚙️ ejecutando.

**Apunta también qué tuviste que corregirle** — eso es lo que más va a valer.

---

**A1 · 🗂️ RAG · BASE**

> *Según el manual del curso, ¿por qué `sort` va siempre antes de `uniq -c`? ¿Qué pasa exactamente
> si me lo salto? Cita el apartado.*

✍️ **Tu respuesta:**


---

**A2 · 🗂️ RAG · BASE**

> *¿Qué hace `tail -n +2` y por qué el manual dice que es «la joya» de ese comando?*

✍️ **Tu respuesta:**


---

**A3 · ⚙️ MÁQUINA · BASE**

Al quitar el `tail -n +2` apareció una línea extraña. **¿Cuál es, y con qué recuento?**

✍️ **Tu respuesta (del comando, no de la IA):**


---

**A4 · 🤖 ASISTENTE · BASE**

> *Explícame la diferencia entre `sort`, `sort -n` y `sort -rn` con un ejemplo de cinco líneas que
> pueda probar. ¿Por qué el 100 puede quedar antes que el 20?*

Pruébalo tú con `printf` o con un fichero pequeño. **No te fíes del ejemplo sin ejecutarlo.**

✍️ **Tu respuesta:**


---

**A5 · 🗂️ RAG · BASE**

> *¿Qué son `NR`, `NF` y `$0` en awk? Dame un ejemplo de cada uno sobre nuestro `ventas.csv`.*

✍️ **Tu respuesta:**


---

**A6 · 🤖 ASISTENTE · COMPLETA**

> *En `awk -F, 'NR>1 {t += $6*$7; n++} END {...}'`, ¿por qué no hay que declarar `t` ni `n` en
> ninguna parte? ¿Qué valor tienen la primera vez que se usan?*

✍️ **Tu respuesta:**


---

**A7 · ⚠️ LA TRAMPA · BASE — hazla, es la más importante del día**

Pregúntale a tu asistente, **sin darle ningún dato**:

> *¿Cuál es la facturación total del fichero ventas.csv de mi curso?*

✍️ **¿Qué te ha contestado?**


✍️ **¿Ha dado un número? ¿De dónde lo ha sacado?**


> 💡 Cualquier respuesta que dé una cifra **sin haber ejecutado nada** es una invención. Puede sonar
> perfecta. Puede tener dos decimales. Y está inventada.
>
> **Esa es la razón de que la columna ⚙️ MÁQUINA exista en este cuaderno.**

### 💰 Enmarca este número

**429.888.864,70 €**

El lunes vas a descubrir algo incómodo sobre él: cuando limpiemos el fichero, **va a cambiar**. Y lo
interesante no será cuánto, sino **hacia dónde**.

✍️ **Tu apuesta, sin medias tintas — ¿subirá o bajará?**

---
---

# PASO 4 · Los códigos de estado del servidor

Cambiamos de fichero. `access.log` **no tiene comas**: se separa por espacios, que ya es el
separador por defecto de awk. Por eso aquí **no lleva `-F`**.

In [ ]:
%%bash
cd ../datasets
awk '{print $9}' access.log | sort | uniq -c | sort -rn

# Esperado: 426839 · 49267 · 15607 · 6372 · 1915

### La contraprueba de segunda vía

¿Y si contamos los errores 500 buscando el texto en vez de mirando la columna? ¿Da lo mismo?

In [ ]:
%%bash
cd ../datasets

echo -n "Por CAMPO   (awk \$9 == 500): "
awk '$9 == 500' access.log | wc -l

echo -n "Por PATRÓN  (grep -c \" 500 \"): "
grep -c " 500 " access.log

---

# 🔍 CONSULTA 2 · Bloque B

---

**B1 · 🗂️ RAG · BASE**

> *Según el manual del curso, ¿qué es cada campo de una línea de log en formato Apache? Cita el
> apartado.*

✍️ **Tu respuesta:**


---

**B2 · 🗂️ RAG · BASE**

> *¿Qué significan las familias 2xx, 3xx, 4xx y 5xx? ¿De quién es la culpa en cada una?*

✍️ **Tu respuesta:**


---

**B3 · ⚙️ MÁQUINA + CRITERIO · BASE**

Compara los dos números de la celda anterior.

✍️ **¿Coinciden? Y coincidan o no, ¿por qué contar « 500 » como texto es una mala forma de medir
esta métrica?**


---

**B4 · 🤖 ASISTENTE · COMPLETA**

> *En un log de servidor, ¿por qué es más robusto extraer el código con `$9` que buscarlo con un
> patrón de texto? Dame un caso concreto en el que el patrón dé un número equivocado.*

✍️ **Tu respuesta:**

---
---

# PASO 5 · Las IP más activas

In [ ]:
%%bash
cd ../datasets
awk '{print $1}' access.log | sort | uniq -c | sort -rn | head -10

### 🛑 PARA AQUÍ

**No ejecutes la siguiente celda todavía.** Compara la primera IP con la segunda.

✍️ **¿Qué acabas de ver? ¿Qué pregunta te haces?**

### La caracterización

Aplicamos el **bucle del analista**:

```
ver raro  →  aislar  →  caracterizar  →  concluir con números
```

In [ ]:
%%bash
cd ../datasets

echo "== ¿Cuántas IP distintas hay en total? (para tener escala) =="
awk '{print $1}' access.log | sort -u | wc -l
# Esperado: 1797

echo ""
echo "== ¿Qué peso tiene la primera IP sobre el total? =="

TOTAL=$(wc -l < access.log)
PRIMERA=$(awk '{print $1}' access.log | sort | uniq -c | sort -rn | head -1 | awk '{print $1}')

awk -v total=$TOTAL -v top=$PRIMERA 'BEGIN {
    printf "  Peticiones totales : %d\n", total
    printf "  La primera IP      : %d\n", top
    printf "  Porcentaje         : %.2f %%\n", top*100/total
}'

# Esperado: 500000 · 42000 · 8.40 %
#
# Dos piezas NUEVAS de awk:
#   -v total=...   mete una variable de bash DENTRO del programa awk
#   BEGIN { }      se ejecuta UNA vez ANTES de leer nada: el gemelo de END
#
# Y la importante: el porcentaje NO está escrito a mano. Sale de los datos.

In [ ]:
%%bash
cd ../datasets

echo "== ¿Qué RUTAS pide? =="
grep "^185\.220\.101\.34 " access.log | awk '{print $7}' | sort | uniq -c | sort -rn | head

echo ""
echo "== ¿Qué RESPUESTAS recibe? =="
grep "^185\.220\.101\.34 " access.log | awk '{print $9}' | sort | uniq -c | sort -rn

---

# 🔍 CONSULTA 3 · Bloque C

---

**C1 · 🤖 ASISTENTE · BASE**

Pégale el patrón `grep "^185\.220\.101\.34 "` y pídele:

> *Explícame este patrón pieza a pieza. ¿Qué hace el `^`? ¿Por qué llevan barra invertida los
> puntos? ¿Qué pasaría si quitara el espacio del final?*

✍️ **Tu respuesta:**


---

**C2 · ⚙️ MÁQUINA · BASE**

✍️ **¿Cuántas rutas distintas pide esa IP? ¿Y qué proporción de sus peticiones acaba en 404?**


---

**C3 · 🗂️ RAG · COMPLETA**

> *Según el material del curso, ¿por qué hay que filtrar los bots ANTES de calcular indicadores de
> tráfico? ¿Qué le pasa a un KPI de visitas si no se hace?*

✍️ **Tu respuesta:**


---

**C4 · 📝 CRITERIO · BASE — tu informe forense en cinco líneas**

No se pregunta a nadie. Lo escribes tú, con tus números.

✍️ **Qué IP · qué porcentaje del tráfico · qué pedía · qué respuestas recibía · qué harías:**

---
---

# PASO 3 · La ciudad que más factura — *esta la escribe la IA y la auditas tú*

Hasta ahora los comandos te los he dado yo. Este no.

### 1 · Pégale esto a tu asistente

```
Tengo un CSV con cabecera:
id_venta,fecha,id_cliente,id_producto,categoria,unidades,precio_unitario,ciudad,canal
1,2025-01-02,84321,P-1042,informatica,2,219.90,Zaragoza,web
2,2025-01-02,45841,P-1361,informatica,3,116.14,Zaragoza,tienda

Quiero la facturación total (unidades × precio_unitario) por ciudad,
ordenada de mayor a menor, con awk y sort.
Explícame el comando pieza a pieza.
```

### 2 · Antes de ejecutar nada, responde por escrito

✍️ **① ¿Qué campo usa para agrupar?**


✍️ **② ¿Dónde acumula?**


✍️ **③ ¿Qué hace el `END`?**


**Si su explicación no te permite responder las tres, pídele que lo reexplique.**
No se ejecuta nada que no se entienda.

### 3 · Pega su comando abajo y ejecútalo

In [ ]:
%%bash
cd ../datasets

# PEGA AQUÍ el comando que te ha dado la IA:



# ---------------------------------------------------------------
# Versión de referencia, para comparar con la tuya:
awk -F, 'NR>1 {c[$8] += $6*$7} END {for (x in c) printf "%16.2f  %s\n", c[x], x}' ventas.csv | sort -rn

# Esperado (top 4):
#   145650275.50  Zaragoza · 59872662.50  Madrid · 51325419.50  Barcelona · 42491335.50  Huesca

### 🔍 La auditoría

**D1 · 📝 CRITERIO · BASE**

✍️ **¿Su comando dio el mismo número que el de referencia? Si no, ¿en qué se diferenciaban?**


---

**D2 · 🔍 MIRA LA COLA DEL RANKING · BASE**

Ahí abajo están `zaragoza`, ` Zaragoza` y la ciudad vacía. Como ciudades separadas. **Con dinero
propio.**

✍️ **¿La IA te avisó de eso? ¿Tenía forma de saberlo?**


> 💡 **No podía.** Le diste el esquema, no los datos. Su comando es correcto y su resultado está
> sucio — y el que tiene que darse cuenta eres tú.
>
> **La suciedad no rompe los comandos: rompe los resultados.**

---

**D3 · 🗂️ RAG · COMPLETA**

> *Según el manual, ¿qué es un array asociativo en awk y en qué se parece a un `GROUP BY` de SQL?*

✍️ **Tu respuesta:**


---

📌 **Guarda ese 145.650.275,50 de Zaragoza.** El martes le vas a pedir a una IA exactamente esta
consulta, te va a dar exactamente este número, y estará mal por más de un millón de euros.

---
---

# `metricas.sh` · De teclear comandos a escribir software

Cuando un análisis merece repetirse, se guarda como **script**. Normalmente se crea con `nano` desde
la Terminal; la celda de abajo hace lo mismo desde el cuaderno para no perder tiempo — pero **lee su
contenido**, porque es el examen.

In [ ]:
%%writefile metricas.sh
#!/bin/bash
# metricas.sh - cuadro de metricas del curso
DATOS=../datasets/ventas.csv
LOG=../datasets/access.log

echo "=========================================="
echo " CUADRO DE METRICAS - $(date +%Y-%m-%d)"
echo "=========================================="

echo ""
echo "-- 1. Operaciones por categoria --"
cut -d, -f5 $DATOS | tail -n +2 | sort | uniq -c | sort -rn

echo ""
echo "-- 2. Facturacion y ticket medio --"
awk -F, 'NR>1 {t += $6*$7; n++} END {printf "Total: %.2f  Ticket: %.2f\n", t, t/n}' $DATOS

echo ""
echo "-- 4. Codigos de estado del servidor --"
awk '{print $9}' $LOG | sort | uniq -c | sort -rn

In [ ]:
%%bash
chmod +x metricas.sh
./metricas.sh > metricas.txt
cat metricas.txt

### 🎯 Lo que acabas de conseguir

Mañana, dentro de un mes o dentro de un año, **una sola orden** reproduce todo el análisis de esta
tarde. `metricas.txt` es el **entregable oficial de este laboratorio**.

Y esa idea —*que el proceso quede montado para repetirse pulsando un botón*— es literalmente el
encargo del proyecto final. Ya la has cumplido en pequeño.

---

### 🔍 CONSULTA 4 · Bloque E — *para el que acabe antes, o para casa*

**E1 · 🗂️ RAG** — *¿Qué es el shebang de un script y qué pasa exactamente si lo omito?*

✍️


**E2 · 🤖 ASISTENTE** — *¿Diferencia entre `./metricas.sh` y `bash metricas.sh`? ¿Cuál necesita
`chmod +x` y por qué?*

✍️


**E3 · 🤖 ASISTENTE** — *¿Por qué Linux me obliga a escribir `./` delante? Explícame qué es el
PATH y qué riesgo de seguridad evita esa norma.*

✍️


**E4 · 🗂️ RAG** — *¿Diferencia entre `>` y `>>`? ¿Cuál de los dos borra sin avisar?*

✍️

---
---

# ★ LAS GRÁFICAS

Un número es **preciso**; una imagen es **convincente** — y tu trabajo va a consistir muchas veces
en convencer a alguien que no va a leer tu tabla.

### El principio que se establece hoy y ya no cambia

```
        LA TERMINAL CALCULA   ·   EL CUADERNO PRESENTA
```

Los números los sigue calculando `awk`. Python **solo dibuja**. El motor calcula, la capa de
presentación presenta, y nunca se mezclan.

## Calentamiento · Una gráfica sin salir de la terminal

In [ ]:
%%bash
cd ../datasets
cut -d, -f9 ventas.csv | tail -n +2 | sort | uniq -c | sort -rn | \
  awk '{printf "%-8s %7d  ", $2, $1; for (i=0; i<$1/10000; i++) printf "#"; print ""}'

---

## Gráfica 1 · Operaciones contra euros

In [ ]:
%%bash
cd ../datasets
awk -F, 'NR>1 {n[$5]++; e[$5] += $6*$7} END {for (c in n) printf "%s,%d,%.2f\n", c, n[c], e[c]}' ventas.csv > ../notebooks/cat.csv
cat ../notebooks/cat.csv

In [ ]:
import matplotlib.pyplot as plt

filas = []
for linea in open("cat.csv"):
    categoria, ops, euros = linea.strip().split(",")
    filas.append((categoria, int(ops), float(euros)))

filas.sort(key=lambda f: -f[2])
nombres     = [f[0] for f in filas]
operaciones = [f[1] for f in filas]
euros       = [f[2] / 1_000_000 for f in filas]

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4.5))

izq.bar(nombres, operaciones, color="#4d9494")
izq.set_title("OPERACIONES por categoría")
izq.set_ylabel("nº de ventas")

der.bar(nombres, euros, color="#026666")
der.set_title("EUROS por categoría")
der.set_ylabel("millones de €")

for eje in (izq, der):
    eje.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 👀 Mira las dos mitades

A la izquierda, cinco barras casi idénticas. A la derecha, **una torre y cuatro tocones**.

**F1 · 📝 CRITERIO · BASE**

✍️ **Tu frase de lectura** — una o dos líneas. No describas el gráfico: di qué **significa**.


---

**F2 · 📝 CRITERIO · BASE**

✍️ **Si le llevaras a dirección solo el gráfico de la izquierda, ¿estarías mintiendo? ¿Por qué?**


---

**F3 · 🗂️ RAG · COMPLETA**

> *El manual del curso dice «contar no es lo mismo que sumar». Búscame dónde lo dice y explícame el
> ejemplo con el que lo justifica.*

✍️

---

## Gráfica 2 · Las ciudades, y la suciedad hecha visible

In [ ]:
%%bash
cd ../datasets
awk -F, 'NR>1 {e[$8] += $6*$7} END {for (c in e) printf "%s|%.2f\n", c, e[c]}' ventas.csv | sort -t'|' -k2 -rn > ../notebooks/ciu.csv
cat ../notebooks/ciu.csv

In [ ]:
import matplotlib.pyplot as plt

limpias = ["Zaragoza", "Madrid", "Barcelona", "Huesca", "Valencia",
           "Sevilla", "Bilbao", "Teruel", "Pamplona", "Logrono"]

ciudades, euros_ciu, colores_ciu = [], [], []
for linea in open("ciu.csv"):
    ciudad, importe = linea.rstrip("\n").split("|")
    ciudades.append(repr(ciudad))          # repr() enseña las comillas y el espacio
    euros_ciu.append(float(importe) / 1_000_000)
    colores_ciu.append("#026666" if ciudad in limpias else "#b45309")

plt.figure(figsize=(9, 6))
plt.barh(ciudades[::-1], euros_ciu[::-1], color=colores_ciu[::-1])
plt.xlabel("millones de €")
plt.title("Facturación por ciudad — en ámbar, lo que no debería existir")
plt.tight_layout()
plt.show()

print(f"{len(ciudades)} barras · en ámbar: {colores_ciu.count('#b45309')}")

### 👀 Mira las barras ámbar

Son **tres**, y son las tres anomalías de ayer, ahora con dinero:

- `' Zaragoza'` — con un espacio delante
- `'zaragoza'` — en minúscula
- `''` — la ciudad **vacía**, que también factura

**G1 · 🐍 CÓDIGO · BASE**

En el código de arriba hay una función que hace visible lo invisible.

✍️ **¿Cuál es, y qué pasaría si la quitáramos?**


---

**G2 · 📝 CRITERIO · BASE**

✍️ **Tu frase de lectura:**


---

**G3 · 🗂️ RAG · COMPLETA**

> *Para el ordenador, «Zaragoza», «zaragoza» y « Zaragoza» son tres ciudades distintas. ¿Qué dice el
> material del curso sobre qué hacer con cada tipo de suciedad: borrar, corregir o conservar?*

✍️ *(pista: es probable que el RAG no lo encuentre. Si es así, apúntalo: es una respuesta honesta y
la veremos el lunes.)*

---

## Gráfica 3 · El bot — y una lección sobre escalas

In [ ]:
%%bash
cd ../datasets
awk '{print $1}' access.log | sort | uniq -c | sort -rn | head -10 > ../notebooks/ips.txt
cat ../notebooks/ips.txt

In [ ]:
import matplotlib.pyplot as plt

ips, peticiones = [], []
for linea in open("ips.txt"):
    n, ip = linea.split()
    ips.append(ip)
    peticiones.append(int(n))

colores_ip = ["#b45309"] + ["#4d9494"] * (len(ips) - 1)

plt.figure(figsize=(9, 5))
plt.barh(ips[::-1], peticiones[::-1], color=colores_ip[::-1])
plt.xlabel("peticiones")
plt.title("Las 10 IP más activas")
plt.tight_layout()
plt.show()

### Y ahora lo mismo, cambiando **una sola línea**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.barh(ips[::-1], peticiones[::-1], color=colores_ip[::-1])
plt.xscale("log")                                     # <-- la única línea nueva
plt.xlabel("peticiones (escala logarítmica)")
plt.title("Las mismas diez IP, con otra escala")
plt.tight_layout()
plt.show()

### ⚖️ La lección que vale más que el gráfico

Los mismos datos. **Ninguna cifra ha cambiado.** Y ahora sí se ven las otras nueve.

**H1 · 📝 CRITERIO · BASE**

✍️ **¿Cuál de las dos escalas usarías para avisar a seguridad? ¿Y cuál para un informe de tráfico
mensual? Justifica las dos.**


---

**H2 · 📝 CRITERIO · COMPLETA**

✍️ **¿Alguna de las dos miente? ¿Alguna de las dos es neutral?**


---

**H3 · 🤖 ASISTENTE · COMPLETA**

> *Tengo un fichero con ventas: categoría, unidades, precio unitario, ciudad, canal y fecha.
> Propón tres gráficos que serían útiles para un comité de dirección, y **di qué decisión permitiría
> tomar cada uno**.*

✍️ **¿Cuál de sus tres propuestas te parece la mejor, y cuál descartarías? ¿Por qué?**


> 💡 Aquí no hay respuesta correcta. Lo que se evalúa es **tu criterio para elegir**, que es
> exactamente lo que la IA no puede poner.

---
---

# 📦 Entregable

Antes de archivar: **`Ctrl+S`**. La celda copia el fichero **guardado en disco**, no lo que ves en
pantalla.

| # | Contenido | ¿Hecho? |
|---|---|---|
| 1 | `metricas.sh` y `metricas.txt` | |
| 2 | La facturación **con su contraprueba en pequeño** — sin las dos vías no cuenta | |
| 3 | Los bloques de preguntas **A, B, C y D** contestados | |
| 4 | Las tres gráficas, **cada una con su frase de lectura** | |
| 5 | El informe forense en cinco líneas | |
| 6 | **La trampa A7**: qué te contestó la IA cuando le preguntaste la facturación sin darle datos | |

> Las respuestas a las 🗂️ y 🤖 valen tanto como los comandos. Y la línea que más vale de todas es
> **«esto tuve que corregírselo»**.

In [ ]:
import shutil, os, glob, json

SESION = 1

# ══════════════════════════════════════════════════════════════════════════
#  GUARDIÁN · ¿está en el disco lo que ves en pantalla?
#
#  Esta celda copia el FICHERO DEL DISCO, no lo que tienes delante. Jupyter
#  guarda solo cada pocos minutos: si archivas antes de un Ctrl+S, entregas
#  el cuaderno SIN tus resultados y el HTML sale sin las gráficas.
# ══════════════════════════════════════════════════════════════════════════

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

cuadernos = [f for f in glob.glob("*lab03*.ipynb") if ".ipynb_checkpoints" not in f]
listo = bool(cuadernos)

if not cuadernos:
    print("  No encuentro el cuaderno de este laboratorio en esta carpeta.")

for cuaderno in cuadernos:
    hechas, total = resultados_en_disco(cuaderno)
    print(f"  {cuaderno}: {hechas} de {total} celdas con resultados en el disco")
    if hechas == 0:
        listo = False

if not listo:
    print()
    print("  " + "=" * 68)
    print("   PARA AQUI. No he archivado nada.")
    print()
    print("   Pulsa  Ctrl+S  (Cmd+S en Mac)  y vuelve a ejecutar ESTA celda.")
    print("   Se archiva el fichero del DISCO, no lo que ves en pantalla.")
    print("  " + "=" * 68)

else:
    os.makedirs("entregables", exist_ok=True)

    # Y los artefactos de este laboratorio, si existen
    piezas = cuadernos + [f for f in ["mi_bitacora.ipynb", "metricas.sh", "metricas.txt"] if os.path.exists(f)]

    print()
    for pieza in piezas:
        shutil.copy(pieza, f"entregables/S{SESION:02d}_{os.path.basename(pieza)}")
        print("  copiado:", pieza)

    anidada = os.path.join("entregables", "entregables")
    if os.path.isdir(anidada):
        shutil.rmtree(anidada)
        print("  limpiado: entregables anidado de una ejecución anterior")

    print()
    print("Contenido de entregables/:")
    for f in sorted(os.listdir("entregables")):
        if f != ".ipynb_checkpoints":
            kb = os.path.getsize(os.path.join("entregables", f)) / 1024
            print(f"    {f:<34} {kb:>8.0f} KB")

In [ ]:
import glob, os

# Exporta a HTML el cuaderno de este laboratorio. El HTML conserva las
# gráficas y las salidas incrustadas: se manda por correo y se ve entero.
for cuaderno in glob.glob("*lab03*.ipynb"):
    if ".ipynb_checkpoints" in cuaderno:
        continue
    salida = f"S{SESION:02d}_" + os.path.splitext(os.path.basename(cuaderno))[0]
    !jupyter nbconvert --to html --output-dir entregables --output {salida} "{cuaderno}"

In [ ]:
import glob, os, re

# ══════════════════════════════════════════════════════════════════════════
#  LA COMPROBACIÓN QUE CIERRA EL CÍRCULO
#
#  Un cuaderno ejecutado deja en el HTML el número de cada celda: [1]:, [2]:...
#  Si no hay ninguno, el HTML NO lleva tus resultados.
#  El TAMAÑO del fichero engaña; este número, no.
# ══════════════════════════════════════════════════════════════════════════

def celdas_ejecutadas(ruta_html):
    h = open(ruta_html, encoding="utf-8", errors="ignore").read()
    return len(re.findall(r"\[[0-9]+\]:", h)), h.count("data:image/png;base64")

htmls = sorted(glob.glob("entregables/*.html"))
vacios = []

if not htmls:
    print("  No hay ningún HTML. ¿Ejecutaste la celda anterior?")

for h in htmls:
    ejecutadas, graficas = celdas_ejecutadas(h)
    kb = os.path.getsize(h) / 1024
    if ejecutadas == 0:
        vacios.append(os.path.basename(h))
    estado = "OK    " if ejecutadas else "VACIO "
    print(f"  {estado} {os.path.basename(h):<32} {ejecutadas:>3} celdas ejecutadas · "
          f"{graficas} gráfica(s) · {kb:.0f} KB")

print()
if not htmls:
    print("  Vuelve a la celda de archivar: sin ella no hay nada que exportar.")
elif vacios:
    print("  " + "=" * 68)
    print("   ESE HTML NO LLEVA TUS RESULTADOS.")
    print("   Pulsa Ctrl+S y repite las DOS celdas anteriores.")
    print("  " + "=" * 68)
else:
    print("  Tu entrega lleva tus resultados. Puedes empaquetarla.")

---
---

# 📖 Para el lunes · lectura recomendada

La teoría te la voy a explicar en clase, como siempre. Pero este manual está escrito para leerse
antes, y hay una diferencia real entre **oír** una explicación por primera vez y **reconocerla**.

Si llegas con estas páginas leídas, mi explicación te sirve para resolver tus dudas en vez de para
presentarte los conceptos — y el tiempo que ganamos se va entero al laboratorio.

```
Manual del Bloque 2 · secciones 4.1 a 4.5

  4.1  El modelo relacional: por qué las tablas ganaron
  4.2  Ficha de herramienta: DuckDB
  4.3  Las seis cláusulas y su orden REAL de ejecución
  4.4  Tipos de datos y el baile del decimal
  4.5  LIMPIO-v1   <--  si solo lees una, que sea esta
```

Son unos veinte minutos. Y si no llegas, no pasa nada: el lunes lo vemos igual.

### Tres preguntas para ir pensando

No hace falta acertar — de hecho, **fallarlas es lo que hace que se queden**. Las responderemos
juntos al empezar el laboratorio.

- ¿`COUNT(*)` sobre `ventas.csv` dirá 1.000.000 o 1.000.001?
- Los censos de ayer (3030 · 2004 · 941 · 465), ¿saldrán idénticos en SQL?
- Al limpiar el fichero, la facturación de hoy —**429.888.864,70 €**— ¿subirá o bajará?

---

### El lunes se acaba la terminal y empieza el idioma de la industria

Y te vas a llevar una sorpresa agradable: **ya sabes SQL a medias**.

| Lo que hiciste en la terminal | Cómo se dice en SQL |
|---|---|
| `head -5 f.csv` | `SELECT * FROM 'f.csv' LIMIT 5` |
| `wc -l` | `SELECT COUNT(*)` |
| `cut -d, -f8` | `SELECT ciudad` |
| <code>sort &#124; uniq -c &#124; sort -rn</code> | `GROUP BY … ORDER BY n DESC` |
| `awk '{t+=$6*$7} END{print t}'` | `SUM(unidades*precio_unitario)` |
| <code>sort -u &#124; wc -l</code> | `COUNT(DISTINCT …)` |

`GROUP BY` no va a ser una palabra mágica: va a ser **tu `sort | uniq -c` con traje**. Y esa es
exactamente la razón de que hayamos empezado por la terminal y no por SQL.